## Exercise - EM Coding

In this exercise, you will be implementing the EM Algorithm in Python for the 1-dimensional case.

a) Read and understand the given code. 
- How many and which data point do you have? 
- How many clusters are you supposed to find?

b) Complete the TODOs in the code. Your initialization for the clusters should give them equal weight, an equal variance of 1 and starting means of 0 and 1 respectively. Run and debug your implementation.

In [84]:
import numpy as np
print(np.__version__)

2.4.4


In [85]:
def normDist(x, mean, variance):
    return np.exp(-(x-mean)*(x-mean) / (2*variance)) / np.sqrt(2 * np.pi * variance)

In [86]:
class Cluster:
    def __init__(self, mean, variance, weight):
        self.mean = mean
        self.variance = variance
        self.weight = weight
        self.m = 0.0

In [89]:
class EM:
    def __init__(self):
        self.k = 2
        self.eps = 0.01
        self.xs = np.array([2.0, 3.0, 9.0, 10.0, 11.0])
        self.rs = np.zeros([self.xs.size, self.k])
        self.iteration = 0
        self.clusters = []

    def start(self):
        # todo: Initialize the k clusters correctly
        for c in range(0, self.k):
            self.clusters.append(Cluster(c, 1.0, 1.0/self.k))

        self.trainModel()

        print("---------------------------------------\n")
        print("Needed", self.iteration, " Iterations")
        for c in range(0, len(self.clusters)):
            cl = self.clusters[c]
            print("    Cluster %2d with mean=%.6f variance=%.6f weight=%6f" %(c, self.clusters[c].mean, cl.variance, cl.weight))
        return

    def trainModel(self):
        # Iterate
        self.iteration = 0
        maxiter = 10
        self.printDebug()
        newL = self.computeL()
        error = self.eps + 1

        while (error >= self.eps) and (self.iteration < maxiter):
            prevL = newL
            self.doEStep()
            self.doMStep()
            self.iteration = self.iteration + 1
            self.printDebug()
            newL = self.computeL()
            error = np.abs(prevL - newL)
        return

    def doEStep(self):
        # TODO: implement the E Step(i.e.compute r - values)
        for d in range(0, len(self.rs)):
            sumNormDist = 0
            sum = 0
            for k in range(0, len(self.rs[d])):
                sum = normDist(self.xs[d], self.clusters[k].mean, self.clusters[k].variance)
                sumNormDist +=  normDist(self.xs[d], self.clusters[k].mean, self.clusters[k].variance)
            for k in range(0, len(self.rs[d])):
                self.rs[d, k] = sum / sumNormDist
        return

    def doMStep(self):
        # TODO: implement the M Step (i.e.update all Cluster - values)
        m = np.zeros(self.k)
        sumM = 0
        #responsibility and cluster weight
        for c in range(0, self.k):
            for r in range(0, len(self.rs[c])):
                m[c] += self.rs[c][r]
                sumM += m[c]
            self.clusters[c].weight = m[c] / sumM

        for c in range(0, self.k):
            for r in range(0, len(self.rs[c])):
                self.clusters[c].mean += self.rs[c][r]*self.xs[r]
            self.clusters[c].mean = self.clusters[c].mean / self.clusters[c].weight

        for c in range(0, len(self.clusters)):
            for r in range(0, len(self.rs[c])):
                self.clusters[c].variance += self.rs[c][r]*(self.xs[r] - self.clusters[c].mean)*(self.xs[r] - self.clusters[c].mean)
            self.clusters[c].variance = self.clusters[c].variance / self.clusters[c].mean            
        
        return

    def computeL(self):
        L = 0
        # TODO: Compute L
        for c in range(0, self.k):
            Lc = 0
            for x in range(0, len(self.xs)):
                Lc += self.clusters[c].weight*normDist(self.xs[x], self.clusters[c].mean, self.clusters[c].variance)
            L = L + np.log(Lc)
            print(L)
        return L

    def printDebug(self):
        print("\nIteration %2d, L= %.6f:" % (self.iteration, self.computeL()))        
        for c in range(0, len(self.clusters)):
            print("    ", end="")
            for i in range(0, self.xs.size):
                print("r[%2d][%2d]=%.6f" % (i, c, self.rs[i, c]), end="  ")
            print("")

        for c in range(0, len(self.clusters)):
            cl = self.clusters[c]
            print("    Cluster %2d with mean=%.6f variance=%.6f weight=%6f" %(c, self.clusters[c].mean, cl.variance, cl.weight))
        return

In [90]:
# main
em = EM()
em.start()


-3.5331959794720684
-5.443868415253917

Iteration  0, L= -5.443868:
    r[ 0][ 0]=0.000000  r[ 1][ 0]=0.000000  r[ 2][ 0]=0.000000  r[ 3][ 0]=0.000000  r[ 4][ 0]=0.000000  
    r[ 0][ 1]=0.000000  r[ 1][ 1]=0.000000  r[ 2][ 1]=0.000000  r[ 3][ 1]=0.000000  r[ 4][ 1]=0.000000  
    Cluster  0 with mean=0.000000 variance=1.000000 weight=0.500000
    Cluster  1 with mean=1.000000 variance=1.000000 weight=0.500000
-3.5331959794720684
-5.443868415253917
-2.1077505474828637
-5.284615104255055

Iteration  1, L= -5.284615:
    r[ 0][ 0]=0.817574  r[ 1][ 0]=0.924142  r[ 2][ 0]=0.999797  r[ 3][ 0]=0.999925  r[ 4][ 0]=0.999972  
    r[ 0][ 1]=0.817574  r[ 1][ 1]=0.924142  r[ 2][ 1]=0.999797  r[ 3][ 1]=0.999925  r[ 4][ 1]=0.999972  
    Cluster  0 with mean=6.131809 variance=3.747093 weight=0.666667
    Cluster  1 with mean=15.889900 variance=20.946625 weight=0.353728
-2.1077505474828637
-5.284615104255055
-0.5444619689929978
-318.42173104681825

Iteration  2, L= -318.421731:
    r[ 0][ 0]=0.03962

C:\Users\jihre\AppData\Local\Temp\ipykernel_11504\1512291169.py:84: RuntimeWarning: divide by zero encountered in log
  L = L + np.log(Lc)
C:\Users\jihre\AppData\Local\Temp\ipykernel_11504\1512291169.py:51: RuntimeWarning: invalid value encountered in scalar divide
  self.rs[d, k] = sum / sumNormDist


In [83]:
# --- EOF ---